<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 21 · A Small Asset Management Library in Python
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook collects executable versions of the core Chapter 21 examples:
- using the `assetlib.core` objects (`Instrument`, `Universe`, `Position`,
  `Portfolio`),
- loading market data via `assetlib.data.MarketData`,
- computing signals and forecast targets via
  `assetlib.signals.SignalEngine`,
- constructing portfolios with `assetlib.portfolio` utilities,
- running a simple backtest with `assetlib.backtest.BacktestEngine`, and
- generating holdings and performance reports with `assetlib.reporting`.


### Imports and Path Setup
We start with standard numerical and plotting libraries, then make sure the
`code/` folder is on `sys.path` so that the `assetlib` package can be imported
+from this notebook.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from assetlib import (
    Instrument,
    MarketData,
    Portfolio,
    Position,
    SignalEngine,
    Universe,
)
from assetlib.backtest import BacktestEngine
from assetlib.portfolio import (
    RiskModel,
    equal_weight,
    estimate_mu_sigma,
    gmv_weights,
    signal_tilt,
)
from assetlib.reporting import exposure_report, performance_report


In [ ]:
LOCAL_EOD = PROJECT_ROOT / "data" / "eod_data.csv"
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"
source = LOCAL_EOD if LOCAL_EOD.exists() else REMOTE_EOD

In [ ]:
prices = pd.read_csv(
    source,
    parse_dates=["Date"],
    index_col="Date",
)
prices.tail()

### Core Objects: Universe, Positions, and Portfolios
Create a small instrument universe and holdings snapshot, mirroring the
example used throughout Part IV. Then wrap the holdings in an
`assetlib.core.Portfolio` so that sector and region exposures are easy to
compute.


In [ ]:
instruments = [
    Instrument(symbol="AAPL", sector="Technology", region="US"),
    Instrument(symbol="NVDA", sector="Technology", region="US"),
    Instrument(symbol="JPM", sector="Financials", region="US"),
    Instrument(symbol="SPY", asset_class="Equity Index", region="Global"),
]
universe = Universe(instruments)
universe.symbols

In [ ]:
holdings = pd.DataFrame(
    {
        "symbol": ["AAPL", "NVDA", "JPM", "SPY"],
        "quantity": [120, 80, 150, 200],
        "price": [180.25, 820.10, 145.30, 520.10],
        "sector": [
            "Technology",
            "Technology",
            "Financials",
            "Equity Index",
        ],
        "region": ["US", "US", "US", "Global"],
        "currency": ["USD", "USD", "USD", "USD"],
    }
)
portfolio = Portfolio.from_holdings_dataframe(holdings)
portfolio.holdings

In [ ]:
sector_weights = portfolio.sector_weights()
region_weights = portfolio.region_weights()
sector_weights, region_weights

### Market Data Helper
Construct a `MarketData` instance from the EOD price table and inspect a small
universe of assets plus the `SPY` benchmark.


In [ ]:
md = MarketData(prices=prices)
universe_symbols = ["AAPL", "JPM", "TLT"]

sub_prices = md.select(universe_symbols + ["SPY"])
sub_prices.tail()

In [ ]:
rets = md.returns(universe_symbols)
rets.tail()

### Signals and Forecast Targets
Use `SignalEngine` to build simple momentum and volatility signals and a
forward-return target, then compute a daily information-coefficient series.


In [ ]:
engine = SignalEngine(market_data=md, universe=universe_symbols)
mom_20d = engine.momentum(window=20)
vol_60d = engine.volatility(window=60)
fwd_5d = engine.forward_returns(horizon=5)

mom_20d.tail(), vol_60d.tail(), fwd_5d.tail()

In [ ]:
z_latest = SignalEngine.zscore(mom_20d).iloc[-1]
ic = SignalEngine.information_coefficient(mom_20d, fwd_5d)

z_latest.round(3), ic.describe().round(3)

### Portfolio Construction Utilities
Estimate annualised expected returns and a covariance matrix for the universe,
then compare equal-weight, minimum-variance, and signal-tilted portfolios.


In [ ]:
risk_model: RiskModel = estimate_mu_sigma(md, universe_symbols)
mu_annual = risk_model.mu_annual
cov_annual = risk_model.cov_annual

mu_annual.round(3)

In [ ]:
w_eq = equal_weight(universe_symbols)
w_gmv = gmv_weights(cov_annual)

w_eq, w_gmv.round(3)

In [ ]:
latest_signal = mom_20d.iloc[-1]
w_sig = signal_tilt(latest_signal, long_only=True, cap_per_asset=0.5)
w_sig.round(3)

### Backtest and Reporting
Construct weekly signal-based target weights, run a simple backtest versus
`SPY`, and generate a compact performance and exposure report.


In [ ]:
weekly = mom_20d.resample("W-FRI").last().dropna()
weekly_weights = weekly.apply(
    lambda row: signal_tilt(row, long_only=True),
    axis=1,
)
weekly_weights.head()

In [ ]:
engine_bt = BacktestEngine(
    market_data=md,
    universe=universe_symbols,
    benchmark="SPY",
)
results = engine_bt.run(target_weights=weekly_weights)
returns_df = results.to_frame()
returns_df.tail()

In [ ]:
portfolio_snapshot = Portfolio.from_holdings_dataframe(holdings)
holdings_table = exposure_report(portfolio_snapshot)
holdings_table

In [ ]:
perf = performance_report(results)
perf.table.round(3)

In [ ]:
perf.summary_text()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
